# NB5 — Modèles neuronaux séquentiels

Notebook des pipelines **P21 à P23**.

## Portée du notebook

Ce notebook compare les architectures neuronales séquentielles classiques sur texte tokenisé :
- `CNN1D` pour les motifs locaux ;
- `BiLSTM` pour les dépendances séquentielles ;
- `BiGRU` comme alternative plus légère.

Ici, **le split train / test existe déjà** dans `../../data/`.
On crée seulement un **jeu de validation stratifié à partir du train**.

In [ ]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn tensorflow openpyxl

from collections import OrderedDict

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    stratified_validation_split,
    evaluate_probability_outputs,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
)

seed_everything(42)
print("TensorFlow :", tf.__version__)

In [ ]:
DATA_DIR = "../../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False

VAL_SIZE_WITHIN_TRAIN = 0.10
RANDOM_STATE = 42

MAX_VOCAB = 20000
MAX_LEN = 50
EMBED_DIM = 128
BATCH_SIZE = 64
EPOCHS = 8
EARLY_STOPPING_PATIENCE = 2

OUTPUT_STEM = "NB5_sequence_neural_models"
RESULTS_DIR = "results"

In [ ]:
df_train, X_train_full, y_train_full, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

X_train, X_val, y_train, y_val = stratified_validation_split(
    X_train_full,
    y_train_full,
    val_size=VAL_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
)

print("Taille train :", len(X_train))
print("Taille validation :", len(X_val))
print("Taille test :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - validation :")
print(y_val.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())

In [ ]:
vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN,
    standardize="lower_and_strip_punctuation",
)

vectorizer.adapt(np.array(X_train))

def build_tf_dataset(texts, labels, batch_size=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((np.array(texts), np.array(labels)))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts), seed=42)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_tf_dataset(X_train, y_train, shuffle=True)
val_ds = build_tf_dataset(X_val, y_val, shuffle=False)
test_ds = build_tf_dataset(X_test, y_test, shuffle=False)

In [ ]:
def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="roc_auc"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )
    return model


def build_cnn1d():
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = layers.Embedding(MAX_VOCAB, EMBED_DIM)(x)
    x = layers.Conv1D(128, 3, activation="relu")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="cnn1d")
    return compile_model(model)


def build_bilstm():
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = layers.Embedding(MAX_VOCAB, EMBED_DIM)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="bilstm")
    return compile_model(model)


def build_bigru():
    inputs = keras.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = layers.Embedding(MAX_VOCAB, EMBED_DIM)(x)
    x = layers.Bidirectional(layers.GRU(64))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="bigru")
    return compile_model(model)


models = OrderedDict({
    "P21_CNN1D": build_cnn1d,
    "P22_BiLSTM": build_bilstm,
    "P23_BiGRU": build_bigru,
})

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    )
]

results = []
histories = {}

for name, builder in models.items():
    print(f"\nEntraînement -> {name}")
    model = builder()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[name] = history.history

    train_scores = model.predict(build_tf_dataset(X_train, y_train, shuffle=False), verbose=0).ravel()
    test_scores = model.predict(test_ds, verbose=0).ravel()
    result = evaluate_probability_outputs(
        name=name,
        y_train=np.array(y_train),
        train_scores=train_scores,
        y_test=np.array(y_test),
        test_scores=test_scores,
        threshold=0.5,
    )
    results.append(result)

results_df = round_results(pd.DataFrame(results))
results_df

In [ ]:
display(results_df)
metric_view = round_results(metric_matrix_from_results(results_df))
display(metric_view)

save_results_bundle(results_df, output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans ./{RESULTS_DIR}")